# Main scan: neighbour-based QA

Loads the LSR-corrected, pair-filtered, cell-combined output from
``main_scan_load.ipynb`` (via ``artifacts/scan_load_state.pkl``) and
runs neighbour-based QA on the calibrated brightness temperature
``T_B(v_LSR)``.  ``W`` is in K * km/s and the neighbour scale floor
is tuned for T_B-space residuals.

Absolute scale is still anchored to ``TCAL11A_ANCHOR_K = 100 K``
from the calibration notebook; a Cygnus-A tie-down can rescale all
``T_B``-derived quantities multiplicatively without re-running QA.

Outputs:
- ``artifacts/main_reobserve.json`` -- list of cells to reobserve
- ``artifacts/spectra_per_session.pdf`` -- per-session T_B(v_LSR)
  grids, with QA-flagged + insufficient-pairs cells excluded


In [ ]:
from utils import (
    compute_cell_metrics,
    neighbor_qa,
    collect_reobserve,
    write_edge_recheck_manifest,
)
from plotters import spectra_per_session_pdf

from pathlib import Path
import json
import pickle
import numpy as np

# --- Hardware ---
HPBW_DEG = 3.4

# --- Per-cell metrics (R-space, no T_B) ---
METRIC_MIN_VALID_CH = 8
METRIC_NOISE_V_MAX_KMS = -100.0
METRIC_SIGNAL_V_LO_KMS = -80.0
METRIC_SIGNAL_V_HI_KMS = 60.0
METRIC_SMOOTH_KERNEL = 5
METRIC_PEAK_MIN_SEP_KMS = 4.0
METRIC_PEAK_PROM_NSIGMA = 2.5
METRIC_MIN_NOISE_CH = 5

# --- Neighbor QA (operates on R; W is in km/s) ---
NEIGHBOR_MAX_SEP_DEG = 2.1
MIN_NEIGHBORS = 2
W_Z_THRESH = 3.0
W_FRAC_THRESH = 0.30
W_SCALE_FLOOR = 1000.0  # K * km/s, tuned for T_B-space QA
PEAK_V_Z_THRESH = 3.0
PEAK_V_ABS_THRESH = 15.0
PEAK_V_MIN_SIGMA = 3.0
PEAK_V_SCALE_FLOOR = 20.0
BIMODAL_MIN_RATIO = 0.68

# --- Edge-clipped recheck (operates on R, not T_B; bandwidth-limited
# detection is independent of temperature calibration) ---
EDGE_KMS = 15.0           # check outer 15 km/s at each end of v_lsr_overlap
EDGE_R_HEIGHT = 0.05      # min baseline-subtracted edge median R to flag

# --- Pair-filter spectrum key (must match main_scan_load.ipynb) ---
PAIR_SPECTRUM_KEY = 'T_B_lsr'

# --- Paths ---
STATE_PATH = Path('artifacts/scan_load_state.pkl')
TCAL_DRIFT_PATH = Path('artifacts/tcal_drift_state.pkl')
REOBSERVE_PATH = Path('artifacts/main_reobserve.json')
EDGE_RECHECK_PATH = Path('artifacts/edge_clipped_recheck.json')
SPECTRA_PDF_PATH = Path('artifacts/spectra_per_session.pdf')

get_ipython().run_line_magic('matplotlib', 'inline')

## 1. Load state from `main_scan_load.ipynb`

In [ ]:
with open(STATE_PATH, 'rb') as f:
    state = pickle.load(f)

cell_combined = state['cell_combined']
viable_pairs_per_cell = state['viable_pairs_per_cell']
cells_insufficient_pairs = state['cells_insufficient_pairs']
v_lsr_overlap = state['v_lsr_overlap']
dv_kms = state['dv_kms']
sessions = state['sessions']
cell_scalars = state['cell_scalars']
F1_MHZ, F2_MHZ = state['lo_pair_mhz']

print(f'Loaded {STATE_PATH} ({STATE_PATH.stat().st_size/1e6:.2f} MB)')
print(f'  {len(cell_combined)} science cells, '
      f'{len(cells_insufficient_pairs)} insufficient-pair cells')
print(f'  dv = {dv_kms:.3f} km/s, v_LSR span '
      f'[{v_lsr_overlap[-1]:.0f}, {v_lsr_overlap[0]:.0f}] km/s, '
      f'{len(sessions)} sessions')
print(f'  {len(cell_scalars)} per-(session, cell) calibration scalars')


## 2. Temperature calibration (T_B from Tcal drift)

Evaluate the ``Tcal_pol(t)`` polynomial from
``main_scan_calibration.ipynb`` (loaded from
``artifacts/tcal_drift_state.pkl``) at each cell's median
observation time, convert the cached per-(session, cell)
``P_on`` / ``P_off`` scalars to ``T_sys``, and multiply
through ``R(v_LSR)`` to get ``T_B(v_LSR)``.

For each cell ``(gl, gb)``:

1. ``Tcal_I(t) = Tcal_pol0(t) + Tcal_pol1(t)`` at the cell's
   median ``t``, clipped to the drift fit range
   ``[t_min, t_max]`` (cells outside are flagged
   ``extrapolated``).
2. Per LO: ``T_sys_I = Tcal_I(t) * P_off / (P_on - P_off)``;
   average the two LOs to a scalar ``T_sys`` per
   (session, cell).
3. Cell-level ``T_sys_cell`` = pair-count-weighted mean across
   sessions where the cell was observed.
4. ``cell['T_B'] = R * T_sys_cell``;
   ``pair['T_B_lsr'] = R_lsr * T_sys`` for each pair's own
   session.

Absolute scale is anchored to ``TCAL11A_ANCHOR_K = 100 K`` from
the calibration notebook -- units are "K relative to the
anchor visit" pending a Cygnus-A tie-down.


In [ ]:
from collections import defaultdict

with open(TCAL_DRIFT_PATH, 'rb') as f:
    drift_state = pickle.load(f)

tcal_poly = drift_state['tcal_poly']
t_min_fit = drift_state['t_min']
t_max_fit = drift_state['t_max']
TCAL_ANCHOR_K = drift_state['anchor']['TCAL11A_ANCHOR_K']
print(f'Loaded {TCAL_DRIFT_PATH}: '
      f"pol0 N={drift_state['tcal_poly_n'][0]}, "
      f"pol1 N={drift_state['tcal_poly_n'][1]}, "
      f'anchor={TCAL_ANCHOR_K:.0f} K')


def session_cell_tsys(sess, gl, gb):
    """(T_sys, extrapolated, n_lo_used) for a (session, gl, gb)
    using the cached P_on / P_off scalars and the Tcal(t)
    polynomial.  Returns (nan, False, 0) if the cell has no
    usable on/off pairs on either LO.
    """
    entry = cell_scalars.get((sess, gl, gb))
    if entry is None:
        return np.nan, False, 0
    t = entry['t_median']
    t_eval = float(np.clip(t, t_min_fit, t_max_fit))
    extrap = (t < t_min_fit) or (t > t_max_fit)
    Tcal_I = float(tcal_poly[0](t_eval) + tcal_poly[1](t_eval))
    tsys_per_lo = []
    for lo in (F1_MHZ, F2_MHZ):
        P_on  = entry[f'P_on_{lo:g}']
        P_off = entry[f'P_off_{lo:g}']
        if not (np.isfinite(P_on) and np.isfinite(P_off)):
            continue
        dP = P_on - P_off
        if dP <= 0:
            continue
        tsys_per_lo.append(Tcal_I * P_off / dP)
    if not tsys_per_lo:
        return np.nan, extrap, 0
    return float(np.mean(tsys_per_lo)), extrap, len(tsys_per_lo)


n_calibrated = 0
n_no_tsys = 0
n_extrap = 0
for (gl, gb), pairs in viable_pairs_per_cell.items():
    sess_counts = defaultdict(int)
    for pr in pairs:
        sess_counts[pr['session']] += 1
    num, den = 0.0, 0.0
    any_extrap = False
    for sess, n_pr in sess_counts.items():
        tsys, extrap, _ = session_cell_tsys(sess, gl, gb)
        if np.isfinite(tsys):
            num += tsys * n_pr
            den += n_pr
        if extrap:
            any_extrap = True
    if den > 0:
        T_sys_cell = num / den
        cell_combined[(gl, gb)]['T_sys'] = float(T_sys_cell)
        cell_combined[(gl, gb)]['T_B'] = cell_combined[(gl, gb)]['R'] * T_sys_cell
        n_calibrated += 1
    else:
        cell_combined[(gl, gb)]['T_sys'] = np.nan
        cell_combined[(gl, gb)]['T_B'] = np.full_like(
            cell_combined[(gl, gb)]['R'], np.nan
        )
        n_no_tsys += 1
    cell_combined[(gl, gb)]['extrapolated'] = bool(any_extrap)
    if any_extrap:
        n_extrap += 1
    for pr in pairs:
        tsys_p, _ext, _ = session_cell_tsys(pr['session'], gl, gb)
        if np.isfinite(tsys_p):
            pr['T_B_lsr'] = pr['R_lsr'] * tsys_p
        else:
            pr['T_B_lsr'] = np.full_like(pr['R_lsr'], np.nan)

tsys_vals = np.array([c['T_sys'] for c in cell_combined.values()
                      if np.isfinite(c.get('T_sys', np.nan))])
print(f'T_B calibration: {n_calibrated} cells calibrated, '
      f'{n_no_tsys} with no usable T_sys, '
      f'{n_extrap} flagged extrapolated')
if tsys_vals.size:
    print(f'  T_sys distribution: median={np.median(tsys_vals):.1f} K, '
          f'IQR [{np.percentile(tsys_vals, 25):.1f}, '
          f'{np.percentile(tsys_vals, 75):.1f}] K, '
          f'range [{tsys_vals.min():.1f}, {tsys_vals.max():.1f}] K')


## 3. Neighbour-based QA on integrated W (T_B-space) and peak velocity

QA operates on the calibrated brightness temperature ``T_B(v_LSR)``
(K) from ``main_scan_load.ipynb`` §7.  ``W`` is in K * km/s and the
neighbour scale floor ``W_SCALE_FLOOR`` is set accordingly (1000
K * km/s, per CLAUDE.md).  Absolute scale is still anchored to the
calibration notebook's 100 K stipulation pending a Cygnus-A
tie-down; residual cell-to-cell consistency is what this QA checks
either way.

Cells flagged ``extrapolated`` in the load notebook (observed
outside the Tcal drift fit window) are passed through QA but
inherit a NaN ``T_B`` (effectively skipped).


In [ ]:
# compute_cell_metrics reads cr['R'] internally; rebind 'R' -> T_B so
# all downstream metrics (W, peak_v, SNR, noise) are in K-space.
cell_combined_tb = {
    k: {**v, 'R': v['T_B']}
    for k, v in cell_combined.items()
    if 'T_B' in v
}

cell_metrics = compute_cell_metrics(
    cell_combined_tb, v_lsr_overlap, dv_kms,
    min_valid_ch=METRIC_MIN_VALID_CH,
    noise_v_max_kms=METRIC_NOISE_V_MAX_KMS,
    signal_v_lo_kms=METRIC_SIGNAL_V_LO_KMS,
    signal_v_hi_kms=METRIC_SIGNAL_V_HI_KMS,
    smooth_kernel=METRIC_SMOOTH_KERNEL,
    peak_min_sep_kms=METRIC_PEAK_MIN_SEP_KMS,
    peak_prom_nsigma=METRIC_PEAK_PROM_NSIGMA,
    min_noise_ch=METRIC_MIN_NOISE_CH,
)
neighbor_cells = neighbor_qa(
    cell_metrics,
    dv_kms=dv_kms,
    hpbw_deg=HPBW_DEG,
    neighbor_max_sep_deg=NEIGHBOR_MAX_SEP_DEG,
    min_neighbors=MIN_NEIGHBORS,
    w_z_thresh=W_Z_THRESH,
    w_frac_thresh=W_FRAC_THRESH,
    w_scale_floor=W_SCALE_FLOOR,
    peak_v_z_thresh=PEAK_V_Z_THRESH,
    peak_v_abs_thresh=PEAK_V_ABS_THRESH,
    peak_v_min_sigma=PEAK_V_MIN_SIGMA,
    peak_v_scale_floor=PEAK_V_SCALE_FLOOR,
    bimodal_min_ratio=BIMODAL_MIN_RATIO,
)

neighbor_flagged = [c for c in neighbor_cells if c['W_flag'] or c['peak_v_flag']]
print(f'Neighbor QA: {len(neighbor_cells)} cells analyzed (on R)')
print(f'  Flags: W={sum(1 for c in neighbor_cells if c["W_flag"])}, '
      f'peak_v={sum(1 for c in neighbor_cells if c["peak_v_flag"])}, '
      f'any={len(neighbor_flagged)}')

if neighbor_flagged:
    print('  Most deviant cells:')

    def _severity(c):
        w = abs(c['W_frac_resid']) if np.isfinite(c['W_frac_resid']) else 0.0
        v = abs(c['peak_v_z']) if np.isfinite(c['peak_v_z']) else 0.0
        return max(w, v)

    for cell in sorted(neighbor_flagged, key=_severity, reverse=True)[:12]:
        print(
            f"    l={cell['gl']:6.2f} b={cell['gb']:3d} "
            f"W_R={cell['W']:+.2f} km/s (frac={cell['W_frac_resid']:+.2f}, z={cell['W_z']:+.2f}) "
            f"v_peak={cell['peak_v']:+.1f} km/s "
            f"(dv={cell['peak_v_resid']:+.1f}, z={cell['peak_v_z']:+.2f}) "
            f"n={cell['neighbor_count']}"
        )

## 4. Reobserve list

In [ ]:
reobs = collect_reobserve(neighbor_cells, cells_insufficient_pairs)
REOBSERVE_PATH.write_text(json.dumps(reobs, indent=2) + '\n')
print(f'Wrote {REOBSERVE_PATH} -- {len(reobs)} cells')
for r in reobs:
    print(f"  l={r['l']:7.2f} b={r['b']:+3d}  ({r['reason']})")

## 4b. Edge-clipped recheck manifest

Some cells have HI line emission whose wings reach the edge of the
current LO pair's frequency-switched usable bandwidth -- the line is
clipped at the boundary of `v_lsr_overlap`. Flag those cells and write
a JSON manifest of repointings at a shifted LO pair so the missing
wing gets covered.

Runs on the dimensionless ratio `R` (still on each cell entry from
the load notebook), not on `T_B`. Bandwidth-limited detection is
independent of temperature calibration -- multiplying `R` by `T_sys`
shifts the height threshold by `T_sys` but doesn't change which
cells are clipped.

LO-pair geometry (`Delta_LO = |F2 - F1|`):

- **Lower pair `(LO0, LO1) = (F1 - Delta_LO, F1)`** -- new overlap
  shifts to lower frequencies, covering more positive v_LSR. Use
  when the line is hot at the most-positive end of `v_lsr_overlap`.
- **Higher pair `(LO2, LO3) = (F2, F2 + Delta_LO)`** -- new overlap
  shifts to higher frequencies, covering more negative v_LSR. Use
  when hot at the most-negative end.

The manifest is consumed by `scripts/main/edges.py` (single-shot
collection driver). Cells are sorted by `|b|` ascending, then by
`gl`.

In [ ]:
edge_manifest = write_edge_recheck_manifest(
    cell_combined,
    v_lsr_overlap=v_lsr_overlap,
    edge_kms=EDGE_KMS,
    r_height_thresh=EDGE_R_HEIGHT,
    excluded=cells_insufficient_pairs,
    current_pair_mhz=(F1_MHZ, F2_MHZ),
    path=EDGE_RECHECK_PATH,
    spectrum_key='R',
    source_notebook='main_scan_qa.ipynb',
)

n_lower  = sum(1 for c in edge_manifest['cells'] if c['suggested_pair'] == 'lower')
n_higher = sum(1 for c in edge_manifest['cells'] if c['suggested_pair'] == 'higher')
print(f'Edge-clipped detection (edge_kms={EDGE_KMS}, '
      f'R height > {EDGE_R_HEIGHT}): '
      f'{len(edge_manifest["cells"])} cells flagged '
      f'({n_lower} lower pair, {n_higher} higher pair)')
print(f'  Lower  pair (LO0, LO1) = {tuple(edge_manifest["lower_pair_mhz"])} MHz')
print(f'  Higher pair (LO2, LO3) = {tuple(edge_manifest["higher_pair_mhz"])} MHz')
print(f'Wrote {EDGE_RECHECK_PATH}')

## 5. Per-session spectra PDF

Renders each session's cells as T_B(v_LSR) grids, with QA-flagged and
insufficient-pair cells excluded.

In [ ]:
qa_flagged_set = {(c['gl'], c['gb']) for c in neighbor_cells
                  if c['W_flag'] or c['peak_v_flag']}
insufficient_set = {(c['l'], c['b']) for c in cells_insufficient_pairs}
excluded_cells = qa_flagged_set | insufficient_set

n_pages = spectra_per_session_pdf(
    SPECTRA_PDF_PATH,
    v_lsr_overlap,
    viable_pairs_per_cell,
    excluded_cells,
    sessions,
    spectrum_key=PAIR_SPECTRUM_KEY,
)
print(f'Saved {SPECTRA_PDF_PATH} ({n_pages} pages); '
      f'excluded {len(excluded_cells)} cells '
      f'({len(qa_flagged_set)} QA + {len(insufficient_set)} insufficient-pairs)')